In [1]:
import torch
import torch.nn as nn

class PaperNetwork4Layers(nn.Module):
    def __init__(self):
        super(PaperNetwork4Layers, self).__init__()
        # Corresponding to W^2, W^3, W^4 with respective input/output dimensions
        self.layer2 = nn.Linear(2, 2)  # W^{[2]} in R^{2x2}, b^{[2]} in R^2
        self.layer3 = nn.Linear(2, 3)  # W^{[3]} in R^{3x2}, b^{[3]} in R^3
        self.layer4 = nn.Linear(3, 2)  # W^{[4]} in R^{2x3}, b^{[4]} in R^2
        self.activation = nn.Sigmoid()

    def forward(self, x):
        a2 = self.activation(self.layer2(x))
        a3 = self.activation(self.layer3(a2))
        a4 = self.activation(self.layer4(a3))  # Overall output F(x)
        return a4

# Example usage with dummy 2D input data
net_4l = PaperNetwork4Layers()
sample_input = torch.tensor([[0.1, 0.3]])
output_4l = net_4l(sample_input)
print("4-Layer Network Output:", output_4l)

4-Layer Network Output: tensor([[0.4893, 0.5033]], grad_fn=<SigmoidBackward0>)


In [2]:
class PaperNetwork5Layers(nn.Module):
    def __init__(self):
        super(PaperNetwork5Layers, self).__init__()
        # Layer dimensions: n1=4, n2=3, n3=4, n4=5, n5=2
        self.layer2 = nn.Linear(4, 3)  # W^{[2]} in R^{3x4}
        self.layer3 = nn.Linear(3, 4)  # W^{[3]} in R^{4x3}
        self.layer4 = nn.Linear(4, 5)  # W^{[4]} in R^{5x4}
        self.layer5 = nn.Linear(5, 2)  # W^{[5]} in R^{2x5}
        self.activation = nn.Sigmoid()

    def forward(self, x):
        a2 = self.activation(self.layer2(x))
        a3 = self.activation(self.layer3(a2))
        a4 = self.activation(self.layer4(a3))
        a5 = self.activation(self.layer5(a4))  # Output layer L=5
        return a5

# Example usage with dummy 4D input data
net_5l = PaperNetwork5Layers()
sample_input_5d = torch.tensor([[0.1, 0.4, 0.5, 0.9]])
output_5l = net_5l(sample_input_5d)
print("5-Layer Network Output:", output_5l)

5-Layer Network Output: tensor([[0.4101, 0.5935]], grad_fn=<SigmoidBackward0>)


In [3]:
import torch
import numpy as np

def train_network():
    # 1. Setup Training Data and Target Values (from Listing 6.1)
    x1 = torch.tensor([0.1, 0.3, 0.1, 0.6, 0.4, 0.6, 0.5, 0.9, 0.4, 0.7], dtype=torch.float32)
    x2 = torch.tensor([0.1, 0.4, 0.5, 0.9, 0.2, 0.3, 0.6, 0.2, 0.4, 0.6], dtype=torch.float32)
    
    # y target matrix: first 5 are category A [1; 0], next 5 are category B [0; 1]
    y = torch.tensor([
        [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 1, 1, 1, 1, 1]
    ], dtype=torch.float32)

    # 2. Initialize Weights and Biases (matching rng(5000) and 0.5*randn scaling)
    torch.manual_seed(5000)
    W2 = 0.5 * torch.randn(2, 2, dtype=torch.float32, requires_grad=True)
    W3 = 0.5 * torch.randn(3, 2, dtype=torch.float32, requires_grad=True)
    W4 = 0.5 * torch.randn(2, 3, dtype=torch.float32, requires_grad=True)
    
    b2 = 0.5 * torch.randn(2, 1, dtype=torch.float32, requires_grad=True)
    b3 = 0.5 * torch.randn(3, 1, dtype=torch.float32, requires_grad=True)
    b4 = 0.5 * torch.randn(2, 1, dtype=torch.float32, requires_grad=True)

    eta = 0.05       # Learning rate
    Niter = 1000000  # Number of stochastic gradient iterations (1e6)

    activation = torch.nn.Sigmoid()

    # 3. Stochastic Gradient Iteration Loop
    for counter in range(1, Niter + 1):
        # Choose a training point at random (MATLAB randi(10) corresponds to indices 0 to 9 in Python)
        k = torch.randint(0, 10, (1,)).item()
        x = torch.tensor([[x1[k]], [x2[k]]], dtype=torch.float32) # 2x1 column vector
        yk = y[:, k].unsqueeze(1) # 2x1 target vector

        # --- Forward Pass ---
        a2 = activation(torch.matmul(W2, x) + b2)
        a3 = activation(torch.matmul(W3, a2) + b3)
        a4 = activation(torch.matmul(W4, a3) + b4)

        # --- Backward Pass (Manual Delta Computation matching Lemma 1) ---
        delta4 = a4 * (1.0 - a4) * (a4 - yk)
        delta3 = a3 * (1.0 - a3) * torch.matmul(W4.t(), delta4)
        delta2 = a2 * (1.0 - a2) * torch.matmul(W3.t(), delta3)

        # --- Gradient Step (In-place parameter updates) ---
        with torch.no_grad():
            W2 -= eta * torch.matmul(delta2, x.t())
            W3 -= eta * torch.matmul(delta3, a2.t())
            W4 -= eta * torch.matmul(delta4, a3.t())
            b2 -= eta * delta2
            b3 -= eta * delta3
            b4 -= eta * delta4

    # 4. Compute Final Cost (Evaluating across all 10 points)
    costvec = []
    with torch.no_grad():
        for i in range(10):
            xi = torch.tensor([[x1[i]], [x2[i]]], dtype=torch.float32)
            yi = y[:, i].unsqueeze(1)
            
            a2_i = activation(torch.matmul(W2, xi) + b2)
            a3_i = activation(torch.matmul(W3, a2_i) + b3)
            a4_i = activation(torch.matmul(W4, a3_i) + b4)
            
            diff = yi - a4_i
            costvec.append(torch.sum(diff ** 2).item())
            
        final_cost = 0.5 * sum(costvec) # Scaled cost function matching paper equation (6)
        print(f"Final Cost after {Niter} iterations: {final_cost}")

if __name__ == "__main__":
    train_network()

Final Cost after 1000000 iterations: 0.836197149910908


In [4]:
import numpy as np

def train_network_numpy():
    # 1. Setup Training Data and Target Values (Listing 6.1 equivalent)
    x1 = np.array([0.1, 0.3, 0.1, 0.6, 0.4, 0.6, 0.5, 0.9, 0.4, 0.7])
    x2 = np.array([0.1, 0.4, 0.5, 0.9, 0.2, 0.3, 0.6, 0.2, 0.4, 0.6])
    
    # Target matrix y (first 5 columns category A [1;0], next 5 category B [0;1])
    y = np.hstack([np.ones((2, 5)) * np.array([[1], [0]]), 
                   np.ones((2, 5)) * np.array([[0], [1]])])

    # 2. Initialize Weights and Biases (matching rng(5000) and 0.5*randn scaling)
    np.random.seed(5000)
    W2 = 0.5 * np.random.randn(2, 2)
    W3 = 0.5 * np.random.randn(3, 2)
    W4 = 0.5 * np.random.randn(2, 3)
    
    b2 = 0.5 * np.random.randn(2, 1)
    b3 = 0.5 * np.random.randn(3, 1)
    b4 = 0.5 * np.random.randn(2, 1)

    eta = 0.05       # Learning rate
    Niter = 1000000  # Number of SGD iterations (1e6)

    # Sigmoid activation function matching activate.m
    def activate(z):
        return 1.0 / (1.0 + np.exp(-z))

    # 3. Stochastic Gradient Iteration Loop
    for counter in range(1, Niter + 1):
        # Random index selection (MATLAB randi(10) -> Python 0 to 9 index)
        k = np.random.randint(0, 10)
        x = np.array([[x1[k]], [x2[k]]]) # 2x1 column vector
        yk = y[:, k:k+1]                 # 2x1 target column vector

        # --- Forward Pass ---
        a2 = activate(np.dot(W2, x) + b2)
        a3 = activate(np.dot(W3, a2) + b3)
        a4 = activate(np.dot(W4, a3) + b4)

        # --- Backward Pass (Lemma 1 Delta calculations) ---
        delta4 = a4 * (1.0 - a4) * (a4 - yk)
        delta3 = a3 * (1.0 - a3) * np.dot(W4.T, delta4)
        delta2 = a2 * (1.0 - a2) * np.dot(W3.T, delta3)

        # --- Gradient Step ---
        W2 -= eta * np.dot(delta2, x.T)
        W3 -= eta * np.dot(delta3, a2.T)
        W4 -= eta * np.dot(delta4, a3.T)
        b2 -= eta * delta2
        b3 -= eta * delta3
        b4 -= eta * delta4

    # 4. Final Cost Evaluation across all 10 points
    costvec = np.zeros(10)
    for i in range(10):
        xi = np.array([[x1[i]], [x2[i]]])
        yi = y[:, i:i+1]
        
        a2_i = activate(np.dot(W2, xi) + b2)
        a3_i = activate(np.dot(W3, a2_i) + b3)
        a4_i = activate(np.dot(W4, a3_i) + b4)
        
        costvec[i] = np.linalg.norm(yi - a4_i, ord=2) ** 2
        
    final_cost = 0.5 * np.sum(costvec)
    print(f"NumPy Implementation - Final Cost: {final_cost}")

if __name__ == "__main__":
    train_network_numpy()

NumPy Implementation - Final Cost: 0.00028738379233442376


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

class PaperCIFARCNN(nn.Module):
    def __init__(self):
        super(PaperCIFARCNN, self).__init__()
        
        # Block 1: Input 32x32x3 -> 16x16x32
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.ReLU()
        )
        
        # Block 2: Input 16x16x32 -> 8x8x32
        self.block2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=5, stride=1, padding=2),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.ReLU()
        )
        
        # Block 3: Input 8x8x32 -> 4x4x64
        self.block3 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2),
            nn.AvgPool2d(kernel_size=2, stride=2),
            nn.ReLU()
        )
        
        # Block 4: Input 4x4x64 -> 1x1x64
        self.block4 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=64, kernel_size=4, stride=1),
            nn.ReLU()
        )
        
        # Block 5: Fully connected layer (Linear) mapping 64 features to 10 classes
        self.fc = nn.Linear(64, 10)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        
        # Flatten from (Batch, 64, 1, 1) to (Batch, 64)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Example validation check with a dummy batch of 2 images (32x32 pixels, 3 channels)
if __name__ == "__main__":
    net = PaperCIFARCNN()
    dummy_images = torch.randn(2, 3, 32, 32)
    outputs = net(dummy_images)
    print("CNN Output Logits Shape (Batch Size, Classes):", outputs.shape)

CNN Output Logits Shape (Batch Size, Classes): torch.Size([2, 10])


In [6]:
import numpy as np
from scipy.optimize import least_squares

# 1. Setup Training Data and Target Values (from Section 2 / Listing 6.1)
x1 = np.array([0.1, 0.3, 0.1, 0.6, 0.4, 0.6, 0.5, 0.9, 0.4, 0.7])
x2 = np.array([0.1, 0.4, 0.5, 0.9, 0.2, 0.3, 0.6, 0.2, 0.4, 0.6])
# Target outputs: first 5 are [1, 0], next 5 are [0, 1]
y_targets = np.hstack([np.ones((5, 2)) * np.array([1, 0]), 
                       np.ones((5, 2)) * np.array([0, 1])])

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# 2. Unpack parameter vector 'p' (23 parameters total) into individual weight matrices and biases
def unpack_params(p):
    # W2: 2x2 (4 params)
    W2 = p[0:4].reshape(2, 2)
    # W3: 3x2 (6 params)
    W3 = p[4:10].reshape(3, 2)
    # W4: 2x3 (6 params)
    W4 = p[10:16].reshape(2, 3)
    # b2: 2x1 (2 params)
    b2 = p[16:18].reshape(2, 1)
    # b3: 3x1 (3 params)
    b3 = p[18:21].reshape(3, 1)
    # b4: 2x1 (2 params)
    b4 = p[21:23].reshape(2, 1)
    return W2, W3, W4, b2, b3, b4

# 3. Define the residual function for least_squares (equivalent to lsqnonlin objective)
def residuals(p):
    W2, W3, W4, b2, b3, b4 = unpack_params(p)
    res_list = []
    
    for i in range(10):
        x = np.array([[x1[i]], [x2[i]]])
        y_true = y_targets[i].reshape(2, 1)
        
        # Forward pass matching equation (4)
        a2 = sigmoid(np.dot(W2, x) + b2)
        a3 = sigmoid(np.dot(W3, a2) + b3)
        a4 = sigmoid(np.dot(W4, a3) + b4)
        
        # Residual per data point: y - F(x) (scaled by sqrt(1/10) implicitly or directly evaluated)
        diff = a4 - y_true
        res_list.append(diff.flatten())
        
    return np.concatenate(res_list)

if __name__ == "__main__":
    # Initialize parameters randomly (matching 0.5 * randn scale)
    np.random.seed(5000)
    p0 = 0.5 * np.random.randn(23)
    
    # Run nonlinear least-squares optimization (Python equivalent to MATLAB lsqnonlin)
    print("Running nonlinear least-squares optimization over 23 parameters...")
    result = least_squares(residuals, p0, method='trf', max_nfev=20000)
    
    print("Optimization completed!")
    print(f"Success: {result.success}")
    print(f"Final Cost (Sum of squared residuals / 2): {0.5 * np.sum(result.fun**2)}")

Running nonlinear least-squares optimization over 23 parameters...


ValueError: cannot reshape array of size 4 into shape (2,1)

In [7]:


def compute_regularized_cost(model, data_loader, lambda_reg=0.01):
    """
    Computes the regularized cost function corresponding to Equation (33) in the paper.
    Includes the standard quadratic error plus the L2 norm penalty of the weight matrices.
    """
    criterion = nn.MSELoss(reduction='sum')
    total_squared_error = 0.0
    N = 0
    
    # 1. Compute standard quadratic cost over data points
    with torch.no_grad():
        for inputs, targets in data_loader:
            outputs = model(inputs)
            # MSELoss in PyTorch calculates (1/N) * sum((y - a)^2). 
            # The paper uses (1/(2N)) * sum(||y - a||^2), so we adjust by multiplying by 0.5
            total_squared_error += 0.5 * criterion(outputs, targets).item()
            N += inputs.size(0)
            
    data_cost = total_squared_error / N

    # 2. Add regularization penalty over all weight matrices W^[l] (excluding biases)
    weight_penalty = 0.0
    for name, param in model.named_parameters():
        if 'weight' in name:  # Matches W^[l] matrices, leaving out biases b^[l]
            weight_penalty += torch.sum(param ** 2)
            
    regularization_term = (lambda_reg / N) * weight_penalty.item()
    
    # Total Cost = Data Cost + Regularization Term (Eq. 33)
    total_cost = data_cost + regularization_term
    return total_cost

# Example usage with the 4-layer network model defined previously
if __name__ == "__main__":
    from __main__ import PaperNetwork4Layers # Assuming previous model
    
    model = PaperNetwork4Layers()
    # Dummy dataset of 10 points
    dummy_inputs = torch.randn(10, 2)
    dummy_targets = torch.randint(0, 2, (10, 2)).float()
    dataset = torch.utils.data.TensorDataset(dummy_inputs, dummy_targets)
    loader = torch.utils.data.DataLoader(dataset, batch_size=10)
    
    reg_cost = compute_regularized_cost(model, loader, lambda_reg=0.01)
    print("Regularized Cost Function Value (Equation 33):", reg_cost)

Regularized Cost Function Value (Equation 33): 0.25680039095878604


In [8]:
import torch
import torch.nn as nn

def compute_softmax_log_loss(model, data_loader):
    """
    Computes the softmax log-loss cost function corresponding to Equation (32) 
    in the paper for K-category classification[cite: 22, 23].
    """
    # CrossEntropyLoss in PyTorch combines LogSoftmax and Negative Log-Likelihood Loss,
    # which directly implements the softmax log-loss function described in Equation (32).
    criterion = nn.CrossEntropyLoss(reduction='sum')
    total_loss = 0.0
    N = 0
    
    with torch.no_grad():
        for inputs, labels in data_loader:
            # inputs: feature vectors, labels: integer class indices (0 to K-1)
            outputs = model(inputs) # Raw network outputs v^[L] (logits)
            total_loss += criterion(outputs, labels).item()
            N += inputs.size(0)
            
    # Average negative log-likelihood over N training points
    softmax_cost = total_loss / N
    return softmax_cost

# Example usage
if __name__ == "__main__":
    # Defining a small model mapping to 10 classes (like CIFAR-10)
    class SoftmaxClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.fc = nn.Linear(4, 10) # 4 features mapping to 10 output classes
            
        def forward(self, x):
            return self.fc(x)

    model = SoftmaxClassifier()
    
    # Dummy data: 10 samples with 4 features each, and true integer class labels (0-9)
    dummy_inputs = torch.randn(10, 4)
    dummy_labels = torch.randint(0, 10, (10,))
    
    dataset = torch.utils.data.TensorDataset(dummy_inputs, dummy_labels)
    loader = torch.utils.data.DataLoader(dataset, batch_size=10)
    
    cost_val = compute_softmax_log_loss(model, loader)
    print("Softmax Log-Loss Cost Value (Equation 32):", cost_val)

Softmax Log-Loss Cost Value (Equation 32): 2.697960090637207


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

def train_with_momentum_and_minibatches():
    # 1. Create a dummy network and dataset (representing feature tensors)
    # Mini-batch size m = 100 as specified in Section 7.5[cite: 25]
    batch_size = 100
    dummy_inputs = torch.randn(1000, 3, 32, 32)
    dummy_labels = torch.randint(0, 10, (1000,))
    
    dataset = torch.utils.data.TensorDataset(dummy_inputs, dummy_labels)
    train_loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    model = nn.Sequential(
        nn.Conv2d(3, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.Flatten(),
        nn.Linear(16 * 32 * 32, 10)
    )
    
    criterion = nn.CrossEntropyLoss()
    
    # 2. Configure Stochastic Gradient with Momentum (Section 7.5)
    # The paper notes learning rate schedules (e.g., eta = 0.05, 0.005, 0.0005) across epochs[cite: 25]
    initial_lr = 0.05
    momentum_factor = 0.9 # Standard momentum moving average of current/past gradients[cite: 24, 25]
    
    optimizer = optim.SGD(model.parameters(), lr=initial_lr, momentum=momentum_factor)
    
    # 3. Training Loop across Epochs
    num_epochs = 3 # Representing a subset of the 45 epochs mentioned in the paper[cite: 24, 25]
    
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for batch_idx, (inputs, targets) in enumerate(train_loader):
            # Zero the parameter gradients
            optimizer.zero_grad()
            
            # Forward pass: compute predicted outputs by passing inputs to the network
            outputs = model(inputs)
            
            # Calculate objective/cost function value (Softmax log-loss via CrossEntropy)
            loss = criterion(outputs, targets)
            
            # Backward pass: compute gradient of the loss with respect to model parameters
            loss.backward()
            
            # Stochastic gradient update step with momentum (mini-batch size m = 100)[cite: 24, 25]
            optimizer.step()
            
            running_loss += loss.item()
            
        print(f"Epoch [{epoch+1}/{num_epochs}], Average Batch Loss: {running_loss / len(train_loader):.4f}")

if __name__ == "__main__":
    train_with_momentum_and_minibatches()

Epoch [1/3], Average Batch Loss: 3.8976
Epoch [2/3], Average Batch Loss: 2.4095
Epoch [3/3], Average Batch Loss: 2.3009
